# Labdata migration stress test

Use this notebook to smoke-test the new labdata-backed behavior analysis implementation before trusting it on live cohorts.

The first sections are synthetic and do not require DataJoint, labdata, or a database connection. The live sections are gated by explicit flags so this notebook is safe to open and run top-to-bottom until you choose to query or write tables.

In [1]:
from __future__ import annotations

from pathlib import Path
import subprocess
import sys

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pyproject.toml").exists():
    candidate = Path.cwd().parent
    if (candidate / "pyproject.toml").exists():
        REPO_ROOT = candidate

for path in [REPO_ROOT, REPO_ROOT / "src"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from behavior_analyses.kernels import build_residual_rate_matrix, fit_psychophysical_kernel
from behavior_analyses.learning import summarize_trialset
from behavior_analyses.psychometrics import cumulative_gaussian, fit_psychometric_labdata


## Synthetic learning metrics

This checks that fake `DecisionTask.TrialSet`-like rows can be summarized without a database.

In [2]:
rng = np.random.default_rng(20260601)
n_trials = 240

response_values = rng.choice(np.array([-1, 0, 1], dtype=float), p=[0.43, 0.07, 0.50], size=n_trials)
correct_values = rng.choice(np.array([0, 1], dtype=float), p=[0.24, 0.76], size=n_trials)

trial_row = {
    "n_trials": n_trials,
    "performance": float(np.nanmean(correct_values)),
    "performance_easy": float(np.nanmean(correct_values[:80])),
    "response_values": response_values,
    "correct_values": correct_values,
    "intensity_values": rng.choice(np.array([-24, -16, -8, -4, 4, 8, 16, 24], dtype=float), size=n_trials),
    "initiation_times": rng.gamma(shape=2.2, scale=0.18, size=n_trials),
    "reaction_times": rng.gamma(shape=2.0, scale=0.12, size=n_trials),
}

learning_summary = summarize_trialset(trial_row)
pd.Series({k: v for k, v in learning_summary.items() if not isinstance(v, np.ndarray)})

n_trials                240.000000
n_with_choice           222.000000
n_correct               182.000000
performance               0.758333
performance_easy          0.800000
mean_initiation_time      0.385213
mean_reaction_time        0.237324
dtype: float64

## Synthetic psychometric fit

This checks the labdata-era response convention directly: `1` is right choice, `-1` is left choice, and `0` is no choice.

In [3]:
stims_per_level = 60
stim_levels = np.array([-30, -20, -12, -6, 0, 6, 12, 20, 30], dtype=float)
stim_values = np.repeat(stim_levels, stims_per_level)
true_params = np.array([1.5, 0.12, 0.04, 0.06])
p_right = cumulative_gaussian(*true_params, stim_values)

responses = np.where(rng.random(stim_values.size) < p_right, 1, -1).astype(float)
no_choice = rng.random(stim_values.size) < 0.03
responses[no_choice] = 0

fit = fit_psychometric_labdata(stim_values, responses)
assert fit is not None
display(pd.Series({k: v for k, v in fit.items() if np.isscalar(v)}))

NameError: name 'cumulative_gaussian' is not defined

In [ ]:
if fit is not None:
    x_grid = np.linspace(fit["stims"].min(), fit["stims"].max(), 300)
    y_grid = fit["function"](*fit["fit_params"], x_grid)

    fig, ax = plt.subplots(figsize=(7, 4))
    ci = np.asarray(fit["p_right_ci"])
    yerr = np.abs(np.vstack([fit["p_right"] - ci[:, 0], ci[:, 1] - fit["p_right"]]))
    ax.errorbar(fit["stims"], fit["p_right"], yerr=yerr, fmt="o", color="black", label="observed")
    ax.plot(x_grid, y_grid, color="tab:blue", label="fit_psychometric")
    ax.axhline(0.5, color="0.75", linewidth=1)
    ax.axvline(fit["bias"], color="tab:blue", linestyle="--", linewidth=1)
    ax.set(xlabel="stimulus", ylabel="p_right", ylim=(-0.03, 1.03), title="Synthetic psychometric stress test")
    ax.legend(frameon=False)
    fig.tight_layout()
else:
    print("No psychometric plot because the package-backed fit was skipped.")

## Synthetic psychophysical kernel

This validates the trial-by-timebin design matrix and logistic-regression kernel on fake `stim_events` arrays.

In [ ]:
n_kernel_trials = 360
timebins = 10
stim_events = []
kernel_responses = []
true_weights = np.linspace(-1.0, 1.3, timebins)

for _ in range(n_kernel_trials):
    event_count = rng.integers(8, 28)
    events = np.sort(rng.uniform(0.0, 1.0, size=event_count))
    x_row, _ = build_residual_rate_matrix([events], [1], timebins=timebins)
    logit = float(x_row[0] @ true_weights / 7.0 + rng.normal(0, 0.35))
    prob = 1 / (1 + np.exp(-logit))
    stim_events.append(events)
    kernel_responses.append(1 if rng.random() < prob else -1)

kernel_fit = fit_psychophysical_kernel(stim_events, kernel_responses, timebins=timebins, cv_splits=5)
print({
    "design_shape": kernel_fit["design_matrix"].shape,
    "mean_cv_score": float(np.mean(kernel_fit["scores"])),
    "n_folds": int(kernel_fit["weights"].shape[0]),
})

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
mean_weights = kernel_fit["weights"].mean(axis=0)
sem_weights = kernel_fit["weights"].std(axis=0, ddof=1) / np.sqrt(kernel_fit["weights"].shape[0])
bin_index = np.arange(timebins)
ax.plot(bin_index, true_weights / np.max(np.abs(true_weights)) * np.max(np.abs(mean_weights)), color="0.55", linestyle="--", label="true shape")
ax.errorbar(bin_index, mean_weights, yerr=sem_weights, fmt="o-", color="tab:green", label="estimated")
ax.axhline(0, color="0.75", linewidth=1)
ax.set(xlabel="time bin", ylabel="kernel weight", title="Synthetic psychophysical kernel")
ax.legend(frameon=False)
fig.tight_layout()

## Import and script checks

These cells inspect the local implementation without touching the database.

In [ ]:
script_paths = [
    REPO_ROOT / "scripts" / "analyses" / "seed_behavior_session_set.py",
    REPO_ROOT / "scripts" / "analyses" / "populate_behavior_tables.py",
    REPO_ROOT / "scripts" / "analyses" / "plot_psychometrics.py",
    REPO_ROOT / "scripts" / "analyses" / "plot_learning_curves.py",
    REPO_ROOT / "scripts" / "analyses" / "plot_psychophysical_kernels.py",
]

for script in script_paths:
    result = subprocess.run([sys.executable, str(script), "--help"], capture_output=True, text=True)
    print(script.name, result.returncode)
    assert result.returncode == 0, result.stderr

In [ ]:
RUN_PLUGIN_IMPORT_CHECK = False

if RUN_PLUGIN_IMPORT_CHECK:
    try:
        from behavior_analyses.io import get_chipmunk_table
        chipmunk_table = get_chipmunk_table()
        print("Chipmunk table import OK:", chipmunk_table)
    except Exception as exc:
        print("Chipmunk table import unavailable in this kernel:", repr(exc))
else:
    print("Skipping Chipmunk plugin import. Flip RUN_PLUGIN_IMPORT_CHECK to True to test plugin loading.")

## Live labdata checks

Set `RUN_LIVE_DB_CHECKS = True` only in an environment with labdata/DataJoint credentials. Set `RUN_DB_WRITES = True` only when you intentionally want to insert or populate user-schema rows.

In [ ]:
RUN_LIVE_DB_CHECKS = False
RUN_DB_WRITES = False

SESSION_SET_ID = "migration_stress_test"
SESSION_SET_NAME = "Migration stress test"
SUBJECTS = ["GRB006", "GRB036"]
TRIALSET = "chipmunk"
MIN_TRIALS = 100

print({
    "run_live_db_checks": RUN_LIVE_DB_CHECKS,
    "run_db_writes": RUN_DB_WRITES,
    "session_set_id": SESSION_SET_ID,
    "subjects": SUBJECTS,
})

In [ ]:
if RUN_LIVE_DB_CHECKS:
    from labdata.schema import DecisionTask, get_user_schema
    from labdata_plugin.analysisschema import (
        BehaviorSessionSet,
        LearningSessionMetrics,
        PsychometricSessionFit,
        PsychometricSubjectFit,
        PsychophysicalKernel,
    )

    schema = get_user_schema()
    print("user schema:", schema)
    for table in [BehaviorSessionSet, LearningSessionMetrics, PsychometricSessionFit, PsychometricSubjectFit, PsychophysicalKernel]:
        print("\n", table.__name__)
        print(table.describe())
else:
    print("Skipping live labdata imports. Flip RUN_LIVE_DB_CHECKS to True to inspect schema definitions against the DB.")

In [ ]:
if RUN_LIVE_DB_CHECKS:
    relation = DecisionTask.TrialSet & {"trialset": TRIALSET}
    if SUBJECTS:
        relation = relation & [{"subject": subject} for subject in SUBJECTS]
    candidate_rows = relation.fetch("KEY", limit=20)
    print("candidate trialsets shown:", len(candidate_rows))
    display(pd.DataFrame(candidate_rows))
else:
    print("Skipping live candidate session query.")

In [ ]:
dry_run_command = [
    sys.executable,
    str(REPO_ROOT / "scripts" / "analyses" / "seed_behavior_session_set.py"),
    "--session-set-id", SESSION_SET_ID,
    "--name", SESSION_SET_NAME,
    "--trialset", TRIALSET,
    "--min-trials", str(MIN_TRIALS),
    "--dry-run",
]
for subject in SUBJECTS:
    dry_run_command.extend(["--subjects", subject])

if RUN_LIVE_DB_CHECKS:
    result = subprocess.run(dry_run_command, cwd=REPO_ROOT, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    assert result.returncode == 0
else:
    print("Dry-run command prepared but not executed:")
    print(" ".join(dry_run_command))

In [ ]:
if RUN_DB_WRITES:
    assert RUN_LIVE_DB_CHECKS, "Set RUN_LIVE_DB_CHECKS=True before RUN_DB_WRITES=True."
    seed_command = [arg for arg in dry_run_command if arg != "--dry-run"]
    result = subprocess.run(seed_command, cwd=REPO_ROOT, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    assert result.returncode == 0
else:
    print("Skipping user-schema inserts. Flip RUN_DB_WRITES to True intentionally to seed BehaviorSessionSet.")

In [ ]:
if RUN_LIVE_DB_CHECKS:
    key = {"session_set_id": SESSION_SET_ID}
    table_counts = {
        "BehaviorSessionSet.Session": len(BehaviorSessionSet.Session & key),
        "BehaviorSessionSet.TrialSet": len(BehaviorSessionSet.TrialSet & key),
        "BehaviorSessionSet.SubjectTrialSet": len(BehaviorSessionSet.SubjectTrialSet & key),
        "LearningSessionMetrics": len(LearningSessionMetrics & key),
        "PsychometricSessionFit": len(PsychometricSessionFit & key),
        "PsychometricSubjectFit": len(PsychometricSubjectFit & key),
        "PsychophysicalKernel": len(PsychophysicalKernel & key),
    }
    display(pd.Series(table_counts, name="rows"))

    pending_counts = {
        "LearningSessionMetrics": len((LearningSessionMetrics.key_source & key) - LearningSessionMetrics),
        "PsychometricSessionFit": len((PsychometricSessionFit.key_source & key) - PsychometricSessionFit),
        "PsychometricSubjectFit": len((PsychometricSubjectFit.key_source & key) - PsychometricSubjectFit),
        "PsychophysicalKernel": len((PsychophysicalKernel.key_source & key) - PsychophysicalKernel),
    }
    display(pd.Series(pending_counts, name="pending"))
else:
    print("Skipping user-schema row counts.")

In [ ]:
if RUN_DB_WRITES:
    populate_command = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "analyses" / "populate_behavior_tables.py"),
        "--session-set-id", SESSION_SET_ID,
    ]
    result = subprocess.run(populate_command, cwd=REPO_ROOT, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    assert result.returncode == 0
else:
    print("Skipping computed-table population. Flip RUN_DB_WRITES to True intentionally to populate results.")

## Live result previews

After seeding/populating, these cells confirm that figures and downstream checks read analysis tables rather than raw notebook state.

In [ ]:
if RUN_LIVE_DB_CHECKS:
    key = {"session_set_id": SESSION_SET_ID}
    psych_rows = (PsychometricSubjectFit & key).fetch(as_dict=True)
    print("pooled psychometric rows:", len(psych_rows))
    if psych_rows:
        display(pd.DataFrame([{k: v for k, v in row.items() if k not in {"stims", "p_right", "p_right_ci", "fit_params"}} for row in psych_rows]))
        row = psych_rows[0]
        stims = np.asarray(row["stims"], dtype=float)
        p = np.asarray(row["p_right"], dtype=float)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(stims, p, "o", color="black")
        ax.set(xlabel="stimulus", ylabel="p_right", title=f"{row.get('subject', SESSION_SET_ID)} pooled psychometric")
        fig.tight_layout()
else:
    print("Skipping live psychometric preview.")

In [ ]:
if RUN_LIVE_DB_CHECKS:
    kernel_rows = (PsychophysicalKernel & {"session_set_id": SESSION_SET_ID}).fetch(as_dict=True)
    print("kernel rows:", len(kernel_rows))
    if kernel_rows:
        row = kernel_rows[0]
        weights = np.asarray(row["weights"], dtype=float)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(weights.mean(axis=0), "o-", color="tab:green")
        ax.axhline(0, color="0.75", linewidth=1)
        ax.set(xlabel="time bin", ylabel="mean weight", title=f"{row.get('subject', SESSION_SET_ID)} kernel")
        fig.tight_layout()
else:
    print("Skipping live kernel preview.")